In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime

SOURCE_PATH     = "/Volumes/bfsi/landing/upi/"
TABLE_NAME      = "bfsi.bronze_layer.upi_transactions"
SCHEMA          = "bronze_layer"
CHECKPOINT_PATH = f"/Volumes/bfsi/landing/upi/_checkpoint/{SCHEMA}_upi_transactions"

BATCH_ID = dbutils.widgets.get("batch_id") \
    if "batch_id" in [w.name for w in dbutils.widgets.getAll()] \
    else f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}"


In [0]:
upi_schema = StructType([
    StructField("txn_ts", TimestampType(), True),
    StructField("upi_txn_id", StringType(), False),
    StructField("payer_vpa", StringType(), True),
    StructField("payee_vpa", StringType(), True),
    StructField("payer_bank", StringType(), True),
    StructField("payee_bank", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("remarks", StringType(), True),
    StructField("device_fingerprint", StringType(), True),
    StructField("ip_address", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])

In [0]:
import hashlib, json

QUARANTINE_PATH = f"/Volumes/bfsi/landing/upi/_quarantine/"

def validate_npci_file(file_path: str) -> dict:
    """
    Validate an NPCI UPI settlement file.
    Expected NPCI header format (first line):
        HDR|<file_date>|<record_count>|<total_amount>|<md5_checksum>
    If no HDR line is present, compute checksum for audit and warn.
    Returns a dict with validation results.
    """
    result = {
        "file": file_path,
        "has_npci_header": False,
        "header_valid": None,
        "record_count_match": None,
        "amount_match": None,
        "checksum_match": None,
        "computed_checksum": None,
        "expected_checksum": None,
        "computed_record_count": None,
        "expected_record_count": None,
        "computed_total_amount": None,
        "expected_total_amount": None,
        "status": "UNKNOWN",
        "errors": []
    }

    # Read raw file content
    raw_lines = (
        spark.read.text(file_path)
        .collect()
    )
    lines = [row.value for row in raw_lines]

    if not lines:
        result["status"] = "FAILED"
        result["errors"].append("Empty file")
        return result

    first_line = lines[0].strip()

    # --- Detect NPCI header ---
    if first_line.upper().startswith("HDR|"):
        result["has_npci_header"] = True
        hdr_parts = first_line.split("|")
        # Expected: HDR|file_date|record_count|total_amount|md5_checksum
        if len(hdr_parts) < 5:
            result["status"] = "FAILED"
            result["errors"].append(f"Malformed NPCI header: expected 5 fields, got {len(hdr_parts)}")
            return result

        try:
            expected_record_count = int(hdr_parts[2])
            expected_total_amount = float(hdr_parts[3])
            expected_checksum     = hdr_parts[4].strip().lower()
        except (ValueError, IndexError) as e:
            result["status"] = "FAILED"
            result["errors"].append(f"Header parse error: {e}")
            return result

        result["expected_record_count"] = expected_record_count
        result["expected_total_amount"] = expected_total_amount
        result["expected_checksum"]     = expected_checksum

        # Data lines: skip HDR line (index 0) and any TRL trailer line
        data_lines = [l for l in lines[1:] if not l.strip().upper().startswith("TRL|")]
        # Also skip column header row if present
        if data_lines and "upi_txn_id" in data_lines[0].lower():
            data_lines = data_lines[1:]
    else:
        # No NPCI header — standard CSV: first line is column header
        data_lines = lines[1:]  # skip column header

    # --- Compute actual metrics ---
    actual_record_count = len(data_lines)
    result["computed_record_count"] = actual_record_count

    # Compute total amount (column index 6 = 'amount' in pipe-delimited)
    total_amount = 0.0
    for line in data_lines:
        parts = line.split("|")
        if len(parts) > 6:
            try:
                total_amount += float(parts[6])
            except (ValueError, IndexError):
                pass
    result["computed_total_amount"] = round(total_amount, 2)

    # Compute MD5 checksum of data portion (all data lines concatenated)
    data_block = "\n".join(data_lines).encode("utf-8")
    computed_checksum = hashlib.md5(data_block).hexdigest()
    result["computed_checksum"] = computed_checksum

    # --- Validate if NPCI header present ---
    if result["has_npci_header"]:
        # Record count
        result["record_count_match"] = (actual_record_count == expected_record_count)
        if not result["record_count_match"]:
            result["errors"].append(
                f"Record count mismatch: expected {expected_record_count}, got {actual_record_count}"
            )

        # Total amount
        result["amount_match"] = (abs(result["computed_total_amount"] - expected_total_amount) < 0.01)
        if not result["amount_match"]:
            result["errors"].append(
                f"Total amount mismatch: expected {expected_total_amount}, got {result['computed_total_amount']}"
            )

        # Checksum
        result["checksum_match"] = (computed_checksum == expected_checksum)
        if not result["checksum_match"]:
            result["errors"].append(
                f"Checksum mismatch: expected {expected_checksum}, got {computed_checksum}"
            )

        result["header_valid"] = all([
            result["record_count_match"],
            result["amount_match"],
            result["checksum_match"]
        ])
        result["status"] = "PASSED" if result["header_valid"] else "FAILED"
    else:
        # No NPCI header: log audit checksum, mark as WARNING
        result["status"] = "WARNING_NO_HEADER"
        result["errors"].append("No NPCI HDR record found — checksum computed for audit only")

    return result


# ── Run validation on all CSV files in source ──
csv_files = [f.path for f in dbutils.fs.ls(SOURCE_PATH) if f.name.endswith(".csv")]

validation_results = []
valid_files   = []
failed_files  = []

for fpath in csv_files:
    vr = validate_npci_file(fpath)
    validation_results.append(vr)

    if vr["status"] == "FAILED":
        failed_files.append(fpath)
    else:
        valid_files.append(fpath)

# ── Print validation summary ──
print("=" * 80)
print("  NPCI FILE HEADER CHECKSUM VALIDATION REPORT")
print("=" * 80)
for vr in validation_results:
    fname = vr["file"].split("/")[-1]
    print(f"\n  File: {fname}")
    print(f"  Status:              {vr['status']}")
    print(f"  Has NPCI Header:     {vr['has_npci_header']}")
    print(f"  Computed Checksum:   {vr['computed_checksum']}")
    print(f"  Computed Records:    {vr['computed_record_count']}")
    print(f"  Computed Amount:     {vr['computed_total_amount']}")
    if vr["has_npci_header"]:
        print(f"  Expected Checksum:   {vr['expected_checksum']}")
        print(f"  Expected Records:    {vr['expected_record_count']}")
        print(f"  Expected Amount:     {vr['expected_total_amount']}")
        print(f"  Record Count Match:  {vr['record_count_match']}")
        print(f"  Amount Match:        {vr['amount_match']}")
        print(f"  Checksum Match:      {vr['checksum_match']}")
    if vr["errors"]:
        for err in vr["errors"]:
            print(f"  ⚠ {err}")
print("\n" + "=" * 80)

# ── Quarantine failed files ──
if failed_files:
    dbutils.fs.mkdirs(QUARANTINE_PATH)
    for fpath in failed_files:
        fname = fpath.split("/")[-1]
        dest  = QUARANTINE_PATH + fname
        dbutils.fs.mv(fpath, dest)
        print(f"  ✗ Quarantined: {fname} → {dest}")

print(f"\n  Valid files: {len(valid_files)} | Failed/Quarantined: {len(failed_files)}")

if not valid_files:
    raise RuntimeError("No valid files to ingest — all files failed NPCI validation.")

# Store results for downstream audit logging
_npci_validation_results = validation_results

In [0]:
# Build lookup: file_name -> (checksum, validation_status)
_checksum_map = {
    vr["file"].split("/")[-1]: (vr["computed_checksum"], vr["status"])
    for vr in _npci_validation_results
}

@F.udf(StringType())
def _get_file_checksum(file_path):
    """Resolve NPCI checksum for a given file path."""
    if file_path:
        fname = file_path.rsplit("/", 1)[-1]
        entry = _checksum_map.get(fname)
        return entry[0] if entry else None
    return None

@F.udf(StringType())
def _get_validation_status(file_path):
    """Resolve NPCI validation status for a given file path."""
    if file_path:
        fname = file_path.rsplit("/", 1)[-1]
        entry = _checksum_map.get(fname)
        return entry[1] if entry else "UNVALIDATED"
    return "UNVALIDATED"

df_upi_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("header", "true")           
    .option("delimiter", "|")           
    .schema(upi_schema)
    .load(SOURCE_PATH)
    # Filter out NPCI header/trailer rows if present
    .filter(~F.col("upi_txn_id").rlike("^(HDR|TRL)"))
    .withColumn("_source_system", F.lit("NPCI_UPI"))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_file_name", F.col("_metadata.file_path"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_npci_file_checksum", _get_file_checksum(F.col("_metadata.file_path")))
    .withColumn("_npci_validation_status", _get_validation_status(F.col("_metadata.file_path")))
    .drop("_metadata")
    )

In [0]:
query = (
    df_upi_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_NAME)
)

query.awaitTermination()

print(f"Auto Loader stream completed for {TABLE_NAME}")

In [0]:
df_verify = spark.table(TABLE_NAME)

print(f"Total UPI transactions ingested: {df_verify.count()}")
display(df_verify.limit(5))